## 2.BUILD CONTENT-BASED RECOMMENDER (Genre + Type)

### import lib

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.ml.feature import CountVectorizer, StringIndexer
from pyspark.ml.linalg import Vectors
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator

spark = SparkSession.builder.appName("Anime-Recommendation").getOrCreate()


### Read data

In [0]:
df1 = spark.read.table("codebasics.silver.silver_anime")
df2 = spark.read.table("codebasics.silver.silver_rating")

###Create content column

In [0]:
anime_content = df1.withColumn(
    "content",
    concat_ws(" ", col("Genre_array"), col("Type"))
)


###Convert text to vector


In [0]:
cv = CountVectorizer(
    inputCol="content",
    outputCol="content_vector"
)

anime_content_array = anime_content.withColumn(
    "Content", split(col("content"), " ")
)
cv_model = cv.fit(anime_content_array.withColumnRenamed("Content", "content"))
anime_vectorized = cv_model.transform(anime_content_array.withColumnRenamed("Content", "content"))

In [0]:
anime_content_array.display()

Anime_id,Anime_name,Genre,Type,Episodes,Rating,Members,Genre_array,Content
32281,Kimi no Na wa,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630,"List(drama, romance, school, supernatural)","List(drama, romance, school, supernatural, Movie)"
5114,Fullmetal Alchemist Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Military, Shounen",Tv,64,9.26,793665,"List(action, adventure, drama, fantasy, magic, military, shounen)","List(action, adventure, drama, fantasy, magic, military, shounen, Tv)"
28977,Gintama,"Action, Comedy, Historical, Parody, Samurai, Sci-Fi, Shounen",Tv,51,9.25,114262,"List(action, comedy, historical, parody, samurai, sci-fi, shounen)","List(action, comedy, historical, parody, samurai, sci-fi, shounen, Tv)"
9253,SteinsGate,"Sci-Fi, Thriller",Tv,24,9.17,673572,"List(sci-fi, thriller)","List(sci-fi, thriller, Tv)"
9969,Gintama039,"Action, Comedy, Historical, Parody, Samurai, Sci-Fi, Shounen",Tv,51,9.16,151266,"List(action, comedy, historical, parody, samurai, sci-fi, shounen)","List(action, comedy, historical, parody, samurai, sci-fi, shounen, Tv)"
32935,Haikyuu Karasuno Koukou VS Shiratorizawa Gakuen Koukou,"Comedy, Drama, School, Shounen, Sports",Tv,10,9.15,93351,"List(comedy, drama, school, shounen, sports)","List(comedy, drama, school, shounen, sports, Tv)"
11061,Hunter x Hunter 2011,"Action, Adventure, Shounen, Super Power",Tv,148,9.13,425855,"List(action, adventure, shounen, super power)","List(action, adventure, shounen, super, power, Tv)"
820,Ginga Eiyuu Densetsu,"Drama, Military, Sci-Fi, Space",Ova,110,9.11,80679,"List(drama, military, sci-fi, space)","List(drama, military, sci-fi, space, Ova)"
15335,Gintama,"Action, Comedy, Historical, Parody, Samurai, Sci-Fi, Shounen",Movie,1,9.1,72534,"List(action, comedy, historical, parody, samurai, sci-fi, shounen)","List(action, comedy, historical, parody, samurai, sci-fi, shounen, Movie)"
15417,Gintama039 Enchousen,"Action, Comedy, Historical, Parody, Samurai, Sci-Fi, Shounen",Tv,13,9.11,81109,"List(action, comedy, historical, parody, samurai, sci-fi, shounen)","List(action, comedy, historical, parody, samurai, sci-fi, shounen, Tv)"


### Naruto-based recommendation (Content-Based)

#### Anime name : Create Variable

In [0]:
input_anime = "Naruto"   # can be any anime name

#### Normalize it

In [0]:
input_anime_clean = input_anime.lower().strip()


#### Use it dynamically ,Not only for the naruto

In [0]:
anime_row = (
    anime_vectorized
    .filter(lower(col("Anime_name")).contains(input_anime_clean))
    .select("content_vector")
    .limit(1)
    .collect()
)

if len(anime_row) == 0:
    raise ValueError(f"{input_anime} not found in dataset")

input_anime_vector = anime_row[0][0]

### Cosine similarity function

In [0]:
def cosine_similarity(v1, v2):
    return float(v1.dot(v2) / (v1.norm(2) * v2.norm(2)))


#### Compute similarity

In [0]:
from pyspark.sql.types import FloatType

cosine_udf = udf(lambda x: cosine_similarity(x, input_anime_vector), FloatType())

anime_similarity = anime_vectorized.withColumn(
    "similarity",
    cosine_udf(col("content_vector"))
)

In [0]:
content_recommendations = (
    anime_similarity
    .orderBy(col("similarity").desc())
    .select("Anime_name", "Genre_array", "Type", "similarity")
)

#display(content_recommendations)
content_recommendations.limit(5).display()


Anime_name,Genre_array,Type,similarity
Boruto Naruto,"List(action, comedy, martial arts, shounen, super power)",Movie,1.0
Naruto Shippuuden,"List(action, comedy, martial arts, shounen, super power)",Movie,1.0
Naruto Shippuuden,"List(action, comedy, martial arts, shounen, super power)",Movie,1.0
Naruto Soyokazeden,"List(action, comedy, martial arts, shounen, super power)",Movie,1.0
Dragon Ball Z,"List(action, adventure, comedy, fantasy, martial arts, shounen, super power)",Movie,0.8944272


In [0]:
df1.write.mode("overwrite").format("delta").saveAsTable("codebasics.gold.Recommendation_system")